# 8.9 同一個基模能切換不同風格版本嗎？


你有同一份文章底稿和兩套編輯註記：一套改得簡短，一套增加例子。要回到簡短版，就應套用整套簡短註記，而不是保留上次一半修改再補另一半。LoRA也有相同問題：同一個基礎模型可以搭配不同修正，但切換時必須知道換的是完整哪一組。

先看[8.8的A、B修正路徑](https://birdhackor.github.io/tiny-perceptron-vlm/8.8.html)。回顧一下，基模保留原矩陣W，adapter（適配器）保存補充路徑的A與B及其設定。有效矩陣是W加上指定adapter的ΔW。本節不用已訓練風格，而是手設兩組不同數字，先驗證切換有沒有正確；數字能換回，不代表它們已經具有文風。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from tiny_perceptron.alignment import LoRALinear

torch.manual_seed(0)
layer = LoRALinear(nn.Linear(4, 3), rank=2, alpha=2)
with torch.no_grad():
    layer.b.fill_(0.1)
    adapter_a = {"a": layer.a.clone(), "b": layer.b.clone()}
    layer.a.fill_(0.2)
    layer.b.fill_(-0.1)
    adapter_b = {"a": layer.a.clone(), "b": layer.b.clone()}
    layer.a.copy_(adapter_a["a"])
    layer.b.copy_(adapter_a["b"])
print("切回A的B平均", round(layer.b.mean().item(), 2))
print("A兩個矩陣都恢復", torch.equal(layer.a, adapter_a["a"]) and torch.equal(layer.b, adapter_a["b"]))

輸入是一個4進3出的線性層，rank為2。`no_grad`讓這段手動設定不被記成訓練計算；`fill_`將矩陣每個元素設成指定數字。`clone()`複製當下內容，否則只是拿同一份可變數字作參照，後面再修改時容易連「備份」一起改了。第一組B全為0.1；第二組A改為0.2，B全為-0.1，確保兩組真的不同。

兩個`copy_`是完整切回的關鍵：分別恢復A與B。輸出第一行平均0.1，第二行`True`，表示逐元素比對都相同。這不是把A的ΔW和B的ΔW相加，也不是訓練了第三種混合風格。手動資料存在記憶體中，程式沒有寫出權重檔；實際保存時也需把兩個矩陣一起存下。

除了數字，還要記錄基模版本、要套用的層名、rank、alpha及縮放公式。同樣的形狀只能說矩陣算得過去，不能保證同一個修正對另一版本的基模有原本意義。例如原基模已改變某個特徵的解讀方式，舊adapter修正的位置也可能不再恰當。

切換還需要恢復縮放設定。例如A、B完全相同，但上次alpha為4，這次訓練時為2，若只載矩陣不恢復alpha，就會把修正放大兩倍。這像拿了正確食譜卻用錯量匙。程式的兩組設定刻意相同，所以只示範矩陣備份與替換；正式檔案需把設定一併保存。也應先核對每個層名都有對應矩陣，漏一層不能只因其他層成功載入就說完整切換。遇到不相容版本，應明確報出差異，讓使用者知道現在套的是哪份基模與註記。

練習刪掉恢復A的那一行，先預測B平均仍是0.1，但「兩個矩陣都恢復」會變`False`，再執行核對。只看B平均，會漏掉一半切換錯誤。
